<a href="https://colab.research.google.com/github/vinipi/ailead_gpumanagement/blob/main/exercise_2_cpuvsgpu_benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [29]:
import time
import platform

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, transforms

FORCE_CPU = True
BATCH_SIZE = 128
EPOCHS = 1
LEARNING_RATE = 1e-3
RANDOM_SEED = 42

torch.manual_seed(RANDOM_SEED)

In [30]:
print("Python platform:", platform.platform())
print("PyTorch version:", torch.__version__)

cuda_available = torch.cuda.is_available()
print("CUDA available:", cuda_available)

if cuda_available:
    print("Device count:",torch.cuda.device_count())
    print("Device name:",torch.cuda.get_device_name(0))
    # YOUR CODE HERE: print the total VRAM of GPU 0 in GB.
    properties = torch.cuda.get_device_properties(0)
    VRAM_GB = properties.total_memory / 1e9
    print("Total VRAM GB:",VRAM_GB)
    pass

# YOUR CODE HERE: choose the device.
# Use CPU when FORCE_CPU is True.
# Otherwise use CUDA if it is available.
device = torch.device("cpu" if FORCE_CPU else "cuda")

print("Using device:", device)
assert device.type in {"cpu", "cuda"}


Python platform: Linux-6.6.122+-x86_64-with-glibc2.35
PyTorch version: 2.11.0+cu128
CUDA available: True
Device count: 1
Device name: Tesla T4
Total VRAM GB: 15.637086208
Using device: cpu


In [31]:
# create a transform that converts images to tensors.
transform = transforms.ToTensor()

# download the Fashion-MNIST training dataset.
train_dataset = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=transform
)

# download the Fashion-MNIST test dataset.
test_dataset = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=transform
)

pin_memory = device.type == "cuda"

# create the training DataLoader.
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

# create the test DataLoader.
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

print("Training samples:", len(train_dataset))
print("Test samples:", len(test_dataset))
print("Batch size:", BATCH_SIZE)
print("pin_memory:", pin_memory)

Training samples: 60000
Test samples: 10000
Batch size: 128
pin_memory: False


In [32]:
X_batch, y_batch = next(iter(train_loader))

print("Image batch shape:", X_batch.shape)
print("Label batch shape:", y_batch.shape)

assert len(train_dataset) == 60_000
assert len(test_dataset) == 10_000
assert X_batch.shape[1:] == (1, 28, 28)
assert y_batch.ndim == 1

Image batch shape: torch.Size([128, 1, 28, 28])
Label batch shape: torch.Size([128])


In [33]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
          nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1),
          nn.ReLU(),
          nn.MaxPool2d(kernel_size=2),
          nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
          nn.ReLU(),
          nn.MaxPool2d(kernel_size=2),
        )

        self.classifier = nn.Sequential(
          nn.Flatten(),
          nn.Linear(32 * 7 * 7, 128),
          nn.ReLU(),
          nn.Linear(128, 10)
        )

    def forward(self, x):
        # YOUR CODE HERE: pass x through features, then classifier.
        x = self.features(x)
        x = self.classifier(x)
        return x

In [34]:
# YOUR CODE HERE: create the model and move it to device.
model = SmallCNN().to(device)

# YOUR CODE HERE: create the loss function.
loss_fn = nn.CrossEntropyLoss()

# YOUR CODE HERE: create the optimizer.
optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

first_parameter = next(model.parameters())
print("Model device:", first_parameter.device)
assert first_parameter.device.type == device.type



Model device: cpu


In [35]:
sample_input = torch.randn(4, 1, 28, 28, device=device)
sample_output = model(sample_input)

print("Sample output shape:", sample_output.shape)
assert sample_output.shape == (4, 10)

Sample output shape: torch.Size([4, 10])


In [36]:
def train_one_epoch(model, train_loader, loss_fn, optimizer, device):
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0
    use_cuda = device.type == "cuda"

    for X_batch, y_batch in train_loader:
        # YOUR CODE HERE: move X_batch and y_batch to device.
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # YOUR CODE HERE: run the forward pass and compute loss.
        outputs = model.forward(X_batch)
        loss = loss_fn(outputs, y_batch)

        # YOUR CODE HERE: clear gradients, backpropagate, and update weights.
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # YOUR CODE HERE: accumulate total loss, correct predictions, and sample count.
        total_loss += loss.item() * X_batch.size(0)
        correct += (outputs.argmax(dim=1) == y_batch).sum().item()
        total += X_batch.size(0)

    average_loss = total_loss / total
    accuracy = correct / total

    return average_loss, accuracy

In [37]:
def evaluate(model, test_loader, loss_fn, device):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0
    use_cuda = device.type == "cuda"

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            # YOUR CODE HERE: move X_batch and y_batch to device.
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            # YOUR CODE HERE: run the forward pass and compute loss.
            outputs = model.forward(X_batch)
            loss = loss_fn(outputs, y_batch)
            # YOUR CODE HERE: accumulate total loss, correct predictions, and sample count.
            total_loss += loss.item() * X_batch.size(0)
            correct += (outputs.argmax(dim=1) == y_batch).sum().item()
            total += X_batch.size(0)

    average_loss = total_loss/total
    accuracy = correct/total

    return average_loss, accuracy

In [38]:
benchmark_results = []

for epoch in range(EPOCHS):
    # YOUR CODE HERE: synchronize CUDA before starting the timer when needed.

    # Synchronize GPU before starting the timer
    if device.type == "cuda":
        torch.cuda.synchronize()

    start_time = time.perf_counter()

    train_loss, train_accuracy = train_one_epoch(
        model=model,
        train_loader=train_loader,
        loss_fn=loss_fn,
        optimizer=optimizer,
        device=device,
    )

    # YOUR CODE HERE: synchronize CUDA before stopping the timer when needed.

    end_time = time.perf_counter()
    epoch_time_seconds = end_time - start_time

    test_loss, test_accuracy = evaluate(
        model=model,
        test_loader=test_loader,
        loss_fn=loss_fn,
        device=device,
    )

    result = {
        "epoch": epoch + 1,
        "device": str(device),
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "test_loss": test_loss,
        "test_accuracy": test_accuracy,
        "epoch_time_seconds": epoch_time_seconds,
    }
    benchmark_results.append(result)

    print(f"Epoch {epoch + 1}")
    print(f"Device: {device}")
    print(f"Train loss: {train_loss:.4f}")
    print(f"Train accuracy: {train_accuracy:.4f}")
    print(f"Test loss: {test_loss:.4f}")
    print(f"Test accuracy: {test_accuracy:.4f}")
    print(f"Epoch time: {epoch_time_seconds:.2f} seconds")

gpu_name = None

if torch.cuda.is_available() and device.type == "cuda":
    # YOUR CODE HERE: get the CUDA GPU name.
    gpu_name = torch.cuda.get_device_name(0)

print("Benchmark summary")
print("-----------------")
print("Platform device:", device)
print("GPU name:", gpu_name)
print("Batch size:", BATCH_SIZE)

for result in benchmark_results:
    print(result)



Epoch 1
Device: cpu
Train loss: 0.5841
Train accuracy: 0.7903
Test loss: 0.4201
Test accuracy: 0.8441
Epoch time: 25.68 seconds
Benchmark summary
-----------------
Platform device: cpu
GPU name: None
Batch size: 128
{'epoch': 1, 'device': 'cpu', 'train_loss': 0.5840860682169596, 'train_accuracy': 0.79035, 'test_loss': 0.42008932209014893, 'test_accuracy': 0.8441, 'epoch_time_seconds': 25.68125961199985}
